# Поблочная загрузка фичей

Отдельный ноутбук только для ручной загрузки фичей блоками. Формат простой: `df_part = utils.get_df(...)`, затем `df = df.merge(...)`, затем `print(...)`. Для блоков с несколькими окнами используется `for month in ...`.

## 0. Imports

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

project_root = Path('/Users/underplums/Documents/work/organic-return-dac')
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import magpie.sql_utils as su
import pyspark.sql.functions as F

from cvm_model.io import State
import cvm_model.utils as utils
import cvm_model.sql_my as sql
from cvm_model.parameters import (
    aud_table,
    fav_omni_features_table,
    preperiod_months,
    uplift_rate_prefix,
    aud_suffix,
)

event_timestamp = datetime(2025, 11, 1)
SAMPLE_N = 10_000
USE_TEMP_AUD_TABLE = True

base_month = event_timestamp.date().replace(day=1).isoformat()
target_month = (pd.Timestamp(base_month) + pd.DateOffset(months=1)).date().isoformat()
feature_date = target_month

base_month, target_month, feature_date

## 1. Connections

In [ ]:
state = State.from_env()
engine = state.credentials.loyalty_gp.sa_engine
session = state.spark.session

engine, session

## 2. Audience

In [ ]:
aud_query_raw = sql.aud_query.format(base_month=base_month)

df = utils.get_df(engine, aud_query_raw).fillna(0)
df = df.astype({col: np.int32 for col in {'contact_id'} & set(df.columns)})

assert len(df) > 0
assert df['contact_id'].nunique() == len(df)

if SAMPLE_N is not None and len(df) > SAMPLE_N:
    df = df.sample(SAMPLE_N, random_state=42).reset_index(drop=True)

print('After audience:', df.shape)
df.head()

## 3. Upload audience to temp table

In [ ]:
if USE_TEMP_AUD_TABLE:
    utils.upload_df(engine, pd.DataFrame(df['contact_id']).astype(int), aud_table)
    aud_query = f'select * from {aud_table}'
else:
    aud_query = aud_query_raw

aud_query

## 4. Target

In [ ]:
df_part = utils.get_df(
    engine,
    sql.target_query.format(
        aud=aud_query,
        target_month=target_month,
    ),
).fillna(0)

int_cols = {'is_dac_next_month', 'target_churn_from_dac', 'target_trns', 'target_login', 'target_dac'} & set(df_part.columns)
df_part = df_part.astype({col: np.int32 for col in int_cols})

if 'target_churn_from_dac' not in df_part.columns and {'target_trns', 'target_login'} <= set(df_part.columns):
    df_part['target_dac'] = df_part['target_trns'] * df_part['target_login']
    df_part['target_churn_from_dac'] = 1 - df_part['target_dac']

df = df.merge(df_part, on='contact_id', how='inner')
print('After target:', df.shape)
display(df['target_churn_from_dac'].value_counts(dropna=False).to_frame('cnt'))
display(df['target_churn_from_dac'].value_counts(normalize=True, dropna=False).to_frame('share'))
df.head()

## 5. Recency

In [ ]:
df_part = utils.get_df(
    engine,
    sql.recency_query.format(
        aud=aud_query,
        date=feature_date,
        month=preperiod_months[0],
    ),
)

df = df.merge(df_part, on='contact_id', how='left')
print('After recency:', df.shape)
df_part.head()

## 6. Refresh audience table

In [ ]:
if USE_TEMP_AUD_TABLE:
    utils.upload_df(engine, pd.DataFrame(df['contact_id']).astype(int), aud_table)
    aud_query = f'select * from {aud_table}'

aud_query

## 7. Favourite OMNI temp table

In [ ]:
query = sql.fav_omni_features_create_query.format(
    aud=aud_query,
    date=feature_date,
    month=preperiod_months[0],
)

create_query = sql.create_table_from_select_query.format(
    table=fav_omni_features_table,
    query=query,
    distribution_col='contact_id',
)

utils.execute_query(engine, f'drop table if exists {fav_omni_features_table}')
utils.execute_query(engine, create_query)

print('Created favourite OMNI temp table:', fav_omni_features_table)

## 8. Cheques

In [ ]:
for month in preperiod_months:
    query = sql.cheque_query if month == preperiod_months[0] else sql.cheque_query_short

    df_part = utils.get_df(
        engine,
        query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After cheques {month}m:', df.shape)

df_part.head()

## 9. App logins

In [ ]:
for month in preperiod_months:
    df_part = utils.get_df(
        engine,
        sql.app_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After app {month}m:', df.shape)

df_part.head()

## 10. OMNI QR

In [ ]:
for month in preperiod_months:
    df_part = utils.get_df(
        engine,
        sql.omni_qr_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After OMNI QR {month}m:', df.shape)

df_part.head()

## 11. OMNI features

In [ ]:
for month in preperiod_months:
    df_part = utils.get_df(
        engine,
        sql.omni_features_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After OMNI features {month}m:', df.shape)

df_part.head()

## 12. Favourite OMNI features

In [ ]:
for month in preperiod_months:
    df_part = utils.get_df(
        engine,
        sql.fav_omni_features_select_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
            fav_omni_features_table=fav_omni_features_table,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After favourite OMNI features {month}m:', df.shape)

df_part.head()

## 13. Unique OMNI features count

In [ ]:
for month in preperiod_months:
    df_part = utils.get_df(
        engine,
        sql.omni_unique_features_count_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After unique OMNI features {month}m:', df.shape)

df_part.head()

## 14. OMNI goals / missions

In [ ]:
for month in preperiod_months[1:]:
    df_part = utils.get_df(
        engine,
        sql.omni_goals_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After OMNI goals {month}m:', df.shape)

df_part.head()

## 15. Accepts

In [ ]:
for month in preperiod_months:
    df_part = utils.get_df(
        engine,
        sql.accept_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After accepts {month}m:', df.shape)

df_part.head()

## 16. Bonuses

In [ ]:
for month in preperiod_months:
    df_part = utils.get_df(
        engine,
        sql.bonus_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After bonuses {month}m:', df.shape)

df_part.head()

## 17. Loyalty level

In [ ]:
for month in preperiod_months:
    df_part = utils.get_df(
        engine,
        sql.level_query.format(
            aud=aud_query,
            date=feature_date,
            month=month,
        ),
    )

    df = df.merge(df_part, on='contact_id', how='left')
    print(f'After loyalty level {month}m:', df.shape)

df_part.head()

## 18. Tendency features

In [ ]:
calc_period = preperiod_months[0]

df['transaction_tendency'] = df['cheque_recency'] / np.maximum(1, df[f'trans_lag_avg_{calc_period}'])
df['login_tendency'] = df['login_recency'] / np.maximum(1, df[f'login_lag_avg_{calc_period}'])
df['omni_qr_tendency'] = df['omni_qr_recency'] / np.maximum(1, df[f'omni_qr_lag_avg_{calc_period}'])
df['omni_features_tendency'] = df['omni_features_recency'] / np.maximum(1, df[f'omni_features_lag_avg_{calc_period}'])

print('After tendency:', df.shape)
df[[
    'transaction_tendency',
    'login_tendency',
    'omni_qr_tendency',
    'omni_features_tendency',
]].head()

## 19. Static features

In [ ]:
df_part = utils.get_df(
    engine,
    sql.static_features_query.format(
        aud=aud_query,
        date=feature_date,
        month=preperiod_months[0],
    ),
)

df = df.merge(df_part, on='contact_id', how='left')
print('After static:', df.shape)
df_part.head()

## 20. DAC history and segmentation

In [ ]:
df_part = utils.get_df(
    engine,
    sql.dac_months_count_query.format(
        aud=aud_query,
        date=feature_date,
        month=preperiod_months[0],
    ),
)

df = df.merge(df_part, on='contact_id', how='left')

if 'dac_age_months' in df.columns:
    df.loc[df['dac_age_months'] == 0, 'dac_age_months'] = 1
if {'dac_months_count', 'dac_age_months'} <= set(df.columns):
    df['dac_months_per_dac_age_ratio'] = df['dac_months_count'] / df['dac_age_months']

print('After DAC history:', df.shape)
df_part.head()

## 21. Uplift rate

In [ ]:
aud_prefix_full = state.settings.preprocess_prefix(event_timestamp) / aud_suffix
uplift_rate_bucket = uplift_rate_prefix.split('//')[1].split('/')[0]
uplift_rate_key = '/'.join(uplift_rate_prefix.split('//')[1].split('/')[1:])
aud_bucket = aud_prefix_full.split('//')[1].split('/')[0]
aud_key = '/'.join(aud_prefix_full.split('//')[1].split('/')[1:])

uplift_rate_table = f's3a://{uplift_rate_bucket}/{uplift_rate_key}'
aud_s3_table = f's3a://{aud_bucket}/{aud_key}'

su.save_to_s3(
    input=aud_query,
    prefix=aud_key,
    input_type='query',
    delete=True,
    broadcast=True,
    bucket=aud_bucket,
)

aud_spark = session.read.parquet(aud_s3_table)
uplift_rate = session.read.parquet(uplift_rate_table).filter(F.col('finish_date') < feature_date)

df_part = aud_spark.join(uplift_rate, 'contact_id', 'inner').toPandas().fillna(0)

visible_uplift_contacts = pd.DataFrame(df_part.groupby('contact_id')['treatment'].nunique())
visible_uplift_contacts = visible_uplift_contacts[visible_uplift_contacts['treatment'] == 2].reset_index()

df_part = df_part[df_part['contact_id'].isin(visible_uplift_contacts['contact_id'].values)].copy()

if 'target_dac' not in df_part.columns and {'target_trns', 'target_login'} <= set(df_part.columns):
    df_part['target_dac'] = df_part['target_trns'] * df_part['target_login']

df_c = pd.DataFrame(
    df_part[df_part['treatment'] == 0]
    .groupby('contact_id')['target_dac']
    .mean()
).reset_index().rename(columns={'target_dac': 'target_dac_0'})

df_t = pd.DataFrame(
    df_part[df_part['treatment'] == 1]
    .groupby('contact_id')['target_dac']
    .mean()
).reset_index().rename(columns={'target_dac': 'target_dac_1'})

df_part = df_c.merge(df_t, on='contact_id')
df_part['uplift_rate'] = df_part['target_dac_1'] - df_part['target_dac_0']
df_part = df_part[['contact_id', 'uplift_rate']]

df = df.merge(df_part, on='contact_id', how='left')
print('After uplift_rate:', df.shape)
df_part.head()

## 22. Fill nulls

In [ ]:
null_cols = [c for c in df.columns if any(s in c for s in ['count', 'sum', 'rto', 'aov'])]
df[null_cols] = df[null_cols].fillna(0)

print('Final df:', df.shape)
df.head()